# Pamoka 01 - Įvadas į DI agentus

Sveiki atvykę į pirmą pamoką kurse **DI agentai pradedantiesiems**!

**DI agentas** yra programa, kuri naudoja didelį kalbos modelį (LLM) kaip savo mąstymo variklį ir gali imtis *veiksmų* tikrame pasaulyje — kviesti API, užklausti duomenų bazių ar vykdyti kodą — siekdama įgyvendinti vartotojo tikslą.

Šiame užrašų knygoje jūs sukursite savo pirmąjį agentą: **kelionių agentą**, kuris rekomenduoja atostogų kryptis. Kelyje išmoksite:

1. Prisijungti prie Azure AI Foundry Agent Service naudodami **Microsoft Agent Framework**.
2. Pateikti agentui **įrankį** — paprastą Python funkciją, kurią jis gali kviesti.
3. Vykdyti agentą ir patikrinti jo atsakymą.
4. Srautinio perdavimo būdu gauti agente atsakymą žodis po žodžio.


## Diegimas

Prieš paleisdami šį užrašų knygelę, įsitikinkite, kad turite:

1. **Azure AI Foundry projektą** su išdiegta pokalbių modeliavimo sistema (pvz., `gpt-4o-mini`).
2. **Prisijungimą per Azure CLI** — terminale paleiskite komandą `az login`.
3. **Nustatytus reikiamus aplinkos kintamuosius:**
   - `AZURE_AI_PROJECT_ENDPOINT` — jūsų Azure AI Foundry projekto galinis taškas.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — įdiegtos modelio pavadinimas.

Žemiau esanti ląstelė įdiegia jums reikalingas Python bibliotekas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Sukuriant savo pirmąjį agentą

Agentui reikia dviejų dalykų:

- **Nurodymų**, kurie pasako *kas jis yra* ir *kaip elgtis* (sistemos įvedimo tekstas).
- **Įrankių** — Python funkcijų, pažymėtų `@tool`, kurias agentas gali kviesti norėdamas gauti informaciją ar atlikti veiksmus.

Žemiau apibrėžiame paprastą įrankį, kuris grąžina populiarių atostogų krypčių sąrašą. Agentas naudos šį įrankį, kai vartotojas paprašys kelionių rekomendacijų.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Srautinės Atsakymų Reakcijos

Dėl interaktyvesnės patirties galite **srautu** gauti agento atsakymą. Vietoj to, kad lauktumėte viso atsakymo, agentas pateikia teksto fragmentus iš karto, kai jie generuojami. Tai ypač naudinga pokalbių sąsajose, kur norite rodyti išvestį realiu laiku.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Santrauka

Šioje pamokoje jūs sužinojote, kaip:

- **Sukurti tiekėją**, kuris jungiasi prie Azure AI Foundry Agent Service per `AzureAIProjectAgentProvider`.
- **Apibrėžti įrankį** naudodami `@tool` dekoratorių, kad agentas galėtų kviesti jūsų Python funkcijas.
- **Paleisti agentą** su vartotojo žinute ir atspausdinti jo atsakymą.
- **Srautiniu būdu perduoti atsakymus** realiuoju laiku.

Kitame pamokoje gilinsimės į agentinius karkasus ir sužinosime, kaip suteikti agentams galingesnius įrankius ir daugiapakopes sprendimų priėmimo galimybes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
